In [1]:
import time
import os
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

from siriuspy.search import IDSearch
from siriuspy.devices import VPU

import epics

# BEFORE IOC CORRECTION
# epics.caput('SI-08SB:ID-IVU18:UN_Reset.HIGH', 1)
# epics.caput('SI-08SB:ID-IVU18:UN_Stop.HIGH', 1)
# epics.caput('SI-08SB:ID-IVU18:UN_Start.HIGH', 1)

# epics.caput('SI-08SB:ID-IVU18:Reset-Cmd.HIGH', 1)
# epics.caput('SI-08SB:ID-IVU18:Abort-Cmd.HIGH', 1)
# epics.caput('SI-08SB:ID-IVU18:KParamChange-Cmd.HIGH', 1)

/opt/mamba/envs/sirius/lib/python3.9/site-packages/epics/ca.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [ ]:
def initialize_vpu(beamline):
    """."""
    # Search ID
    devnamevpu = IDSearch.conv_beamline_2_idname(beamline=beamline)
    vpu = VPU(devname=devnamevpu)

    # Disable beamline control
    vpu.cmd_beamline_ctrl_disable()
    print("beamline control: ", vpu.is_beamline_ctrl_enabled)
    return vpu


def move_vpu_gap(vpu: VPU, gap, timeout, verbose=False):
    """."""
    vpu.set_gap(gap)
    time.sleep(0.5)
    print("Gap-RB {:.3f} mm".format(vpu.gap)) if verbose else 0
    if vpu.cmd_move_start(timeout):
        time.sleep(0.5)
        print("Undulator is moving...") if verbose else 0
        while vpu.is_moving:
            time.sleep(0.1)
            print(
                "Current gap {:.3f} mm.".format(vpu.gap_mon), end="\r"
            ) if verbose else 0
        print("Gap {:.3f} mm reached.".format(vpu.gap)) if verbose else 0
        return True
    else:
        print("Error while cmd_move_start.")
        return False


## Search devnames

In [ ]:
devnamevpu_carnauba = IDSearch.conv_beamline_2_idname(beamline="CARNAUBA")
devnamevpu_caterete = IDSearch.conv_beamline_2_idname(beamline="CATERETE")
devnamevpu_caterete

'SI-07SP:ID-VPU29'

## Initialize CATERETE VPU

In [ ]:
vpu_caterete = initialize_vpu(beamline="CATERETE")

Could not set value of SI-07SP:ID-VPU29:BeamLineCtrlEnbl-Sel
beamline control:  False


In [8]:
vpu_caterete.PROPERTIES_DEFAULT

('PitchOffsetMinPos-Cte',
 'MoveStart-Cmd',
 'Reset-Cmd',
 'MoveAcc-SP',
 'KParamParked-Cte',
 'CenterOffset-Mon',
 'BeamLineCtrlEnbl-Sel',
 'Moving-Mon',
 'PitchOffset-Mon',
 'KParamMaxVelo-SP',
 'TaperMinPos-Cte',
 'Taper-SP',
 'KParam-Mon',
 'MoveVelo-RB',
 'KParamMaxVelo-RB',
 'CenterOffsetVelo-Mon',
 'BeamLineCtrlEnbl-Sts',
 'CenterOffset-RB',
 'TaperVelo-Mon',
 'CenterOffset-SP',
 'Taper-RB',
 'PitchOffsetVelo-Mon',
 'KParam-RB',
 'PeriodLength-Cte',
 'MoveAcc-RB',
 'CenterOffsetMinPos-Cte',
 'Abort-Cmd',
 'PitchOffsetMaxPos-Cte',
 'KParam-SP',
 'TaperMaxPos-Cte',
 'MoveVelo-SP',
 'CenterOffsetMaxPos-Cte',
 'Taper-Mon')

In [ ]:
print("Period: {:.1f} mm".format(vpu_caterete.period_length))
print("Polarization mon: {:}".format(vpu_caterete.polarization_mon))
print("Connection:", vpu_caterete.connected)
vpu_caterete.disconnected_pvnames

In [ ]:
target_fields = np.array([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])

gaps = np.linspace(80, 9.7, 101)

# VPU CAT Halbach coefficients
Br = 1.3
a = 1.9586
b = -3.4126
c = 0.1496

bs = Br * a * np.exp(b * (gaps / 29) + c * (gaps / 29) ** 2)


mask = np.isclose(bs[:, None], target_fields[None, :], rtol=0.05, atol=1e-8)
idxs = np.where(mask.any(axis=1))[0]

target_gaps = gaps[idxs]

print("Fields [T]: ")
print(bs[idxs].round(3))
print("\nGaps [mm]: ")
print(target_gaps.round(2))

Fields [T]: 
[0.101 0.2   0.295 0.404 0.512 0.601 0.704 0.763 0.827]

Gaps [mm]: 
[28.68 22.35 18.84 16.03 13.92 12.51 11.11 10.4   9.7 ]


## Gap mov tests

In [ ]:
vpu_caterete.set_gap_speed(0.5)

In [ ]:
move_vpu_gap(vpu_caterete, gap=24, timeout=3, verbose=True)

In [ ]:
gaps0 = np.arange(24, 3, -1)
gaps1 = gaps0[::-1]
gaps = np.concatenate((gaps0, gaps1))
gaps

In [ ]:
for i, gap in enumerate(gaps):
    sucess = move_vpu_gap(vpu_caterete, gap=gap, timeout=3, verbose=True)
    if not sucess:
        break

## Random movement

In [ ]:
timeout = 3
deltat = 180
t0 = time.time()
while datetime.now().hour < 14:
    if deltat >= 180:
        t0 = time.time()
        gap = 5 * np.random.random(1) + 20
        speed = 0.08 * np.random.random(1) + 0.02
        vpu_caterete.set_gap_speed(speed, timeout)
        time.sleep(0.5)
        sucess = move_vpu_gap(vpu_caterete, gap=gap, timeout=timeout, verbose=True)
        if not sucess:
            break
    t = time.time()
    deltat = t - t0
    time.sleep(1)
    print("waiting...", end="\r")

# Taper mov tests

# Pitch mov tests

# Center Offset tests